# 10 — Validate first-isolate selection

## Goal

The first reporting model selects one culture per patient, organism, cohort and **phenotype**. That works for phenotype-specific analysis, but it is not an overall antibiogram population: the same patient can contribute more than once when resistance changes over time.

I keep that model for the separate MDRO analysis and validate a second, phenotype-independent selection for the conventional susceptibility mart. The overall rule is the earliest culture per patient, source release, cohort and organism reporting group. ARMD does not provide a verified isolate identifier, so this is a project-defined first-**culture** policy.

Run the SQL cells using the existing Databricks-connected `%sql` notebook setup. The overall-isolate checks require `dbt build --select +int_overall_first_reporting_susceptibility` to have completed.

## 1. Check the existing phenotype-specific branch

### Grain and earliest-record selection

First, verify the current `int_first_reporting_isolate` grain. The phenotype is part of its key, so these checks should pass even when patients have multiple phenotype records.

In [1]:
%sql
with duplicates as (

    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        reporting_phenotype,
        count(*) as records

    from workspace.micro_dev.int_first_reporting_isolate

    group by
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        reporting_phenotype

    having count(*) > 1

)

select
    (
        select count(*)
        from workspace.micro_dev.int_first_reporting_isolate
    ) as selected_rows,

    (
        select count(distinct first_reporting_isolate_key)
        from workspace.micro_dev.int_first_reporting_isolate
    ) as unique_keys,

    (
        select count(*)
        from duplicates
    ) as duplicate_reporting_groups,

    case
        when (
            select count(*)
            from duplicates
        ) = 0

        and (
            select count(*)
            from workspace.micro_dev.int_first_reporting_isolate
        ) = (
            select count(distinct first_reporting_isolate_key)
            from workspace.micro_dev.int_first_reporting_isolate
        )

        then 'pass'
        else 'fail'
    end as validation_status;

,selected_rows,unique_keys,duplicate_reporting_groups,validation_status
0,87643,87643,0,pass


In [2]:
%sql
with expected_ranked as (

    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        reporting_phenotype,

        reporting_culture_organism_key,
        culture_time_shifted,
        culture_key,

        row_number() over (
            partition by
                source_version,
                cohort_id,
                patient_key,
                organism_reporting_group,
                reporting_phenotype

            order by
                culture_time_shifted,
                culture_key,
                reporting_culture_organism_key
        ) as expected_rank

    from workspace.micro_dev.int_reporting_culture_organism

    where culture_time_shifted is not null
),

expected_first as (

    select *
    from expected_ranked
    where expected_rank = 1

),

comparison as (

    select
        e.source_version,
        e.cohort_id,
        e.patient_key,
        e.organism_reporting_group,
        e.reporting_phenotype,

        e.reporting_culture_organism_key
            as expected_reporting_record,

        a.reporting_culture_organism_key
            as actual_reporting_record,

        e.culture_time_shifted
            as expected_time,

        a.culture_time_shifted
            as actual_time

    from expected_first e

    full outer join workspace.micro_dev.int_first_reporting_isolate a
        on e.source_version = a.source_version
       and e.cohort_id = a.cohort_id
       and e.patient_key = a.patient_key
       and e.organism_reporting_group =
           a.organism_reporting_group
       and e.reporting_phenotype =
           a.reporting_phenotype

)

select
    count(*) as reporting_groups,

    sum(
        case
            when expected_reporting_record
                 = actual_reporting_record
                then 1
            else 0
        end
    ) as correctly_selected,

    sum(
        case
            when expected_reporting_record
                 <> actual_reporting_record
              or expected_reporting_record is null
              or actual_reporting_record is null
                then 1
            else 0
        end
    ) as selection_mismatches,

    case
        when sum(
            case
                when expected_reporting_record
                     <> actual_reporting_record
                  or expected_reporting_record is null
                  or actual_reporting_record is null
                    then 1
                else 0
            end
        ) = 0
            then 'pass'
        else 'fail'
    end as validation_status

from comparison;

,reporting_groups,correctly_selected,selection_mismatches,validation_status
0,87643,87643,0,pass


### Susceptibility reconciliation after phenotype-level consolidation

A reporting record can combine multiple source organism labels from the same culture. The AST model rechecks each antibiotic across those labels and retains conflicts instead of selecting an arbitrary interpretation.

In [3]:
%sql
select
    resolution_status,
    reconciliation_reason,

    count(*) as endpoint_records,

    sum(
        case
            when source_organisms_with_endpoint > 1
                then 1
            else 0
        end
    ) as multi_source_endpoints

from workspace.micro_dev.int_first_reporting_susceptibility

group by
    resolution_status,
    reconciliation_reason

order by
    resolution_status,
    reconciliation_reason;

,resolution_status,reconciliation_reason,endpoint_records,multi_source_endpoints
0,conflict,discordant_source_results,278,278
1,conflict,source_endpoint_conflict,4078,118
2,missing,missing_only,1359,1
3,resolved,resolved_consistent_result,1055013,1911
4,resolved,resolved_with_missing_source_endpoint,2,2


In [4]:
%sql
with key_check as (

    select
        count(*) as row_count,
        count(distinct reporting_susceptibility_key) as unique_key_count
    from workspace.micro_dev.int_first_reporting_susceptibility

),

logic_check as (

    select
        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility is null
                    then 1
                else 0
            end
        ) as resolved_with_null_result,

        sum(
            case
                when resolution_status in ('conflict', 'missing')
                 and susceptibility is not null
                    then 1
                else 0
            end
        ) as unresolved_with_result

    from workspace.micro_dev.int_first_reporting_susceptibility

)

select
    k.row_count,
    k.unique_key_count,

    l.resolved_with_null_result,
    l.unresolved_with_result,

    case
        when k.row_count = k.unique_key_count
         and l.resolved_with_null_result = 0
         and l.unresolved_with_result = 0
            then 'pass'
        else 'fail'
    end as validation_status

from key_check k
cross join logic_check l;

,row_count,unique_key_count,resolved_with_null_result,unresolved_with_result,validation_status
0,1060730,1060730,0,0,pass


## 2. Measure the overlap created by phenotype-specific selection

Now remove the phenotype from the grouping and count how many first records the existing model contributes per patient and organism. This is the reason we need a separate overall population. For example, the earlier urine *E. coli* check found 38,247 phenotype-first records for 37,104 patient–organism groups; the query below recalculates that from the current build.

In [5]:
%sql
with patient_groups as (

    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        count(*) as phenotype_first_records

    from workspace.micro_dev.int_first_reporting_isolate

    group by
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group

)

select
    cohort_id,
    organism_reporting_group,
    count(*) as unique_patient_organism_groups,
    sum(phenotype_first_records)
        as current_phenotype_first_records,
    sum(
        case
            when phenotype_first_records > 1 then 1
            else 0
        end
    ) as patients_in_multiple_phenotypes

from patient_groups

group by
    cohort_id,
    organism_reporting_group

order by
    patients_in_multiple_phenotypes desc,
    cohort_id,
    organism_reporting_group;

,cohort_id,organism_reporting_group,unique_patient_organism_groups,current_phenotype_first_records,patients_in_multiple_phenotypes
0,urine,Escherichia coli,37104,38247,1132
1,respiratory,Pseudomonas aeruginosa,1889,2089,200
2,respiratory,Staphylococcus aureus,3430,3587,157
3,urine,Klebsiella pneumoniae,6428,6540,110
4,blood,Escherichia coli,3022,3065,42
5,urine,Enterobacter cloacae complex,1122,1161,35
6,blood,Staphylococcus aureus,1737,1768,31
7,urine,Pseudomonas aeruginosa,1956,1981,25
8,urine,Klebsiella aerogenes,924,936,12
9,urine,Klebsiella oxytoca,1018,1029,11


## 3. Validate the overall first-culture model

`int_overall_first_reporting_isolate` groups all reporting phenotype records from one culture and organism before selecting the earliest culture. CRE/MDR-positive cultures remain eligible for the overall population.

### One record per patient/cohort/organism

Expect zero duplicate keys, zero duplicate natural-key groups, and no empty source-key arrays.

In [6]:
%sql
with duplicate_groups as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        count(*) as n
    from workspace.micro_dev.int_overall_first_reporting_isolate
    group by
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group
    having count(*) > 1
)
select
    count(*) as overall_records,
    count(distinct overall_reporting_isolate_key) as unique_overall_keys,
    count(*) - count(distinct overall_reporting_isolate_key) as duplicate_keys,
    (select count(*) from duplicate_groups) as duplicate_groups,
    sum(case when source_culture_organism_keys is null
              or size(source_culture_organism_keys) = 0
             then 1 else 0 end) as records_missing_source_keys
from workspace.micro_dev.int_overall_first_reporting_isolate;

,overall_records,unique_overall_keys,duplicate_keys,duplicate_groups,records_missing_source_keys
0,85818,85818,0,0,0


### Compare overall and phenotype-first population counts

Overall counts should match distinct patients within each release, cohort and reporting organism. Phenotype-first counts can be larger; their difference quantifies the repeated contributions avoided by the overall policy. Every row should show `population_check = pass`.

In [7]:
%sql
with phenotype_first as (
    select
        source_version,
        cohort_id,
        organism_reporting_group,
        count(*) as phenotype_first_records,
        count(distinct patient_key) as unique_patients
    from workspace.micro_dev.int_first_reporting_isolate
    group by
        source_version, cohort_id, organism_reporting_group
),
overall_first as (
    select
        source_version,
        cohort_id,
        organism_reporting_group,
        count(*) as overall_first_records
    from workspace.micro_dev.int_overall_first_reporting_isolate
    group by
        source_version, cohort_id, organism_reporting_group
)
select
    coalesce(p.source_version, o.source_version) as source_version,
    coalesce(p.cohort_id, o.cohort_id) as cohort_id,
    coalesce(p.organism_reporting_group, o.organism_reporting_group)
        as organism_reporting_group,
    p.unique_patients,
    p.phenotype_first_records,
    o.overall_first_records,
    p.phenotype_first_records - o.overall_first_records
        as phenotype_extra_records,
    case
        when p.unique_patients = o.overall_first_records
         and p.phenotype_first_records >= o.overall_first_records
            then 'pass'
        else 'mismatch'
    end as population_check
from phenotype_first p
full outer join overall_first o
    on p.source_version = o.source_version
   and p.cohort_id = o.cohort_id
   and p.organism_reporting_group = o.organism_reporting_group
order by
    case
        when p.unique_patients = o.overall_first_records
         and p.phenotype_first_records >= o.overall_first_records
            then 1
        else 0
    end,
    phenotype_extra_records desc,
    cohort_id,
    organism_reporting_group;

,source_version,cohort_id,organism_reporting_group,unique_patients,phenotype_first_records,overall_first_records,phenotype_extra_records,population_check
0,2025-04-11,urine,Escherichia coli,37104,38247,37104,1143,pass
1,2025-04-11,respiratory,Pseudomonas aeruginosa,1889,2089,1889,200,pass
2,2025-04-11,respiratory,Staphylococcus aureus,3430,3587,3430,157,pass
3,2025-04-11,urine,Klebsiella pneumoniae,6428,6540,6428,112,pass
4,2025-04-11,blood,Escherichia coli,3022,3065,3022,43,pass
5,2025-04-11,urine,Enterobacter cloacae complex,1122,1161,1122,39,pass
6,2025-04-11,blood,Staphylococcus aureus,1737,1768,1737,31,pass
7,2025-04-11,urine,Pseudomonas aeruginosa,1956,1981,1956,25,pass
8,2025-04-11,urine,Klebsiella aerogenes,924,936,924,12,pass
9,2025-04-11,urine,Klebsiella oxytoca,1018,1029,1018,11,pass


### Independently select the earliest culture

I rebuild the first-culture ranking directly from `int_reporting_culture_organism`, ignoring phenotype. This is an independent check of the selected culture, not just a uniqueness test. Expect `selection_mismatches = 0`.

In [8]:
%sql
with observed_cultures as (
    select distinct
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        culture_key,
        culture_time_shifted
    from workspace.micro_dev.int_reporting_culture_organism
    where culture_time_shifted is not null
),
independent_rank as (
    select
        *,
        row_number() over (
            partition by
                source_version,
                cohort_id,
                patient_key,
                organism_reporting_group
            order by culture_time_shifted, culture_key
        ) as rn
    from observed_cultures
),
expected as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        culture_key
    from independent_rank
    where rn = 1
),
actual as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        culture_key
    from workspace.micro_dev.int_overall_first_reporting_isolate
),
comparison as (
    select
        e.culture_key as expected_culture,
        a.culture_key as selected_culture
    from expected e
    full outer join actual a
        on e.source_version = a.source_version
       and e.cohort_id = a.cohort_id
       and e.patient_key = a.patient_key
       and e.organism_reporting_group = a.organism_reporting_group
)
select
    count(*) as compared_groups,
    sum(case when expected_culture = selected_culture
             then 0 else 1 end) as selection_mismatches
from comparison;

,compared_groups,selection_mismatches
0,85818,0


## 4. Validate overall-culture AST reconciliation

The selected culture can contain multiple source organism records from different phenotype categories. `int_overall_first_reporting_susceptibility` combines their antibiotic endpoints and preserves missing and conflicting interpretations.

The key count must match the row count. Resolved endpoints must have a result; unresolved endpoints must not.

In [9]:
%sql
select
    count(*) as endpoint_rows,
    count(distinct overall_reporting_susceptibility_key) as unique_keys,
    sum(case when resolution_status = 'resolved'
              and susceptibility is null
             then 1 else 0 end) as resolved_with_null,
    sum(case when resolution_status <> 'resolved'
              and susceptibility is not null
             then 1 else 0 end) as unresolved_with_result
from workspace.micro_dev.int_overall_first_reporting_susceptibility;

,endpoint_rows,unique_keys,resolved_with_null,unresolved_with_result
0,1033825,1033825,0,0


### Inspect reconciliation outcomes

This breakdown shows how much of the overall AST is clean versus missing or conflicting, including endpoints consolidated from multiple source labels. It is useful for tracing changes after the new overall selection.

In [10]:
%sql
select
    resolution_status,
    reconciliation_reason,
    count(*) as endpoint_rows,
    sum(case when source_organisms_with_endpoint > 1
             then 1 else 0 end) as multi_source_endpoints
from workspace.micro_dev.int_overall_first_reporting_susceptibility
group by resolution_status, reconciliation_reason
order by resolution_status, reconciliation_reason;

,resolution_status,reconciliation_reason,endpoint_rows,multi_source_endpoints
0,conflict,discordant_source_results,470,470
1,conflict,source_endpoint_conflict,3694,151
2,missing,missing_only,1347,2
3,resolved,resolved_consistent_result,1028311,2104
4,resolved,resolved_with_missing_source_endpoint,3,3


## Result

The phenotype-first branch remains available for the MDRO workflow. The new overall branch selects one earliest culture per patient and organism within a cohort, then reconciles AST across all source labels from that culture. Notebook 11 checks how those records feed the final susceptibility mart. The first-isolate rule is specific to this project and should not be presented as complete CLSI M39 compliance.